# SparkClient: Configuring Advanced Batch Job Options

This notebook demonstrates how to pass Kubernetes-native specifications (custom names, labels, annotations, node selectors, and tolerations) to Spark applications using `SparkClient`.

### What you will learn:
1. Configuring Kubernetes options (`Name`, `Labels`, `Annotations`, `NodeSelector`, `Toleration`).
2. Submitting a remote Spark `FileJob` with custom pod placement and scheduling rules.
3. Polling and waiting for the job to reach completion.
4. Querying the underlying Kubernetes Custom Resource to verify options propagation.
5. Deleting the Spark job resource.

## 1. Imports and Client Initialization

Import client classes and option specifications from `kubeflow.spark` and initialize `SparkClient`.

In [ ]:
import os
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import (
    Annotations,
    FileJob,
    Labels,
    Name,
    NodeSelector,
    SparkClient,
    SparkJobStatus,
    Toleration,
)

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Define Custom Options and Submit Job

Specify Kubernetes pod options including custom job name, labels, annotations, node selectors, and tolerations, then submit the `FileJob`.

In [ ]:
REMOTE_JOB = "https://raw.githubusercontent.com/kubeflow/sdk/main/examples/spark/spark_job.py"
custom_job_name = f"batch-job-options-{uuid.uuid4().hex[:8]}"

options = [
    Name(custom_job_name),
    Labels({"app": "spark", "team": "ml"}),
    Annotations({"owner": "kubeflow", "environment": "dev"}),
    NodeSelector({"kubernetes.io/os": "linux"}),
    Toleration(
        key="dedicated",
        operator="Equal",
        value="spark",
        effect="NoSchedule",
    ),
]

print(f"Submitting Spark job: {custom_job_name}...")
job_name = client.submit_job(
    job=FileJob(
        file_source=REMOTE_JOB,
        args=["10"],
    ),
    options=options,
)

print(f"Job submitted successfully: {job_name}")

## 3. Wait for Job Completion

Block until the job reaches the completed status.

In [ ]:
print(f"Waiting for {job_name} to complete...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.COMPLETED},
    timeout=300,
)

print("Job completed successfully.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

## 4. Inspect Applied Kubernetes Options

Query the underlying `SparkApplication` custom resource from the Kubernetes API to verify labels, annotations, driver/executor node selectors, and tolerations.

In [ ]:
response = client.backend.custom_api.get_namespaced_custom_object(
    group="sparkoperator.k8s.io",
    version="v1beta2",
    namespace=client.backend.namespace,
    plural="sparkapplications",
    name=job_name,
)

metadata = response["metadata"]
spec = response["spec"]

print(f"Name: {metadata.get('name')}")
print(f"Labels: {metadata.get('labels')}")
print(f"Annotations: {metadata.get('annotations')}")

driver = spec.get("driver", {})
print(f"Driver NodeSelector: {driver.get('nodeSelector')}")
print(f"Driver Tolerations: {driver.get('tolerations')}")

executor = spec.get("executor", {})
print(f"Executor NodeSelector: {executor.get('nodeSelector')}")
print(f"Executor Tolerations: {executor.get('tolerations')}")

## 5. Clean Up Resources

Delete the completed Spark application resource.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deletion requested successfully.")